In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

SEED = 42
DATA_DIR = "./" 

TARGET, ID = "Exited", "id"
    
train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")

def engineer_features(df):
    df = df.copy()
    df['IsBalanceZero'] = (df['Balance'] == 0).astype(int)
    df['Balance_Age_ratio'] = df['Balance'] / df['Age']
    df['CreditScore_Age_ratio'] = df['CreditScore'] / df['Age']
    return df

X_train_eng = engineer_features(train)
X_test_eng  = engineer_features(test)

X_train_eng = X_train_eng.drop(columns=[ID, 'Surname'])
X_test_eng = X_test_eng.drop(columns=[ID, 'Surname'])

X = pd.get_dummies(X_train_eng.drop(columns=[TARGET]), drop_first=True)
y = X_train_eng[TARGET]
Xte = pd.get_dummies(X_test_eng, drop_first=True)

X, Xte = X.align(Xte, join='left', axis=1, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
Xte_scaled = scaler.transform(Xte)

model = RandomForestClassifier(random_state=SEED, n_estimators=100, max_depth=8)
model.fit(X_train_scaled, y_train)

val_pred = model.predict(X_val_scaled)
acc = accuracy_score(y_val, val_pred)
print(f"검증 데이터 정확도 (Accuracy): {acc:.4f}")

importances = pd.Series(model.feature_importances_, index=X.columns)
print("\n모델이 가장 중요하게 생각한 상위 5개 변수:")
print(importances.sort_values(ascending=False).head(5))

test_pred_proba = model.predict_proba(Xte_scaled)[:, 1]

submission = pd.DataFrame({ID: test[ID], TARGET: test_pred_proba})
submission.to_csv("submission_basic_rf_proba.csv", index=False)
print("\n확률값이 포함된 제출 파일 생성 완료")

검증 데이터 정확도 (Accuracy): 0.8596

모델이 가장 중요하게 생각한 상위 5개 변수:
Age                      0.271683
NumOfProducts            0.256343
CreditScore_Age_ratio    0.131784
IsActiveMember           0.097257
Geography_Germany        0.060265
dtype: float64

확률값이 포함된 제출 파일 생성 완료


In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
DATA_DIR = "./" 
TARGET, ID = "Exited", "id"

train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")

def engineer_features(df):
    df = df.copy()
    df['IsBalanceZero'] = (df['Balance'] == 0).astype(int)
    df['Balance_Age_ratio'] = df['Balance'] / df['Age']
    df['CreditScore_Age_ratio'] = df['CreditScore'] / df['Age']
    return df

X_train_eng = engineer_features(train)
X_test_eng  = engineer_features(test)

X_train_eng = X_train_eng.drop(columns=[ID, 'Surname'])
X_test_eng = X_test_eng.drop(columns=[ID, 'Surname'])

X = pd.get_dummies(X_train_eng.drop(columns=[TARGET]), drop_first=True)
y = X_train_eng[TARGET]
Xte = pd.get_dummies(X_test_eng, drop_first=True)
X, Xte = X.align(Xte, join='left', axis=1, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
Xte_scaled = scaler.transform(Xte)

# 트리의 반복 횟수는 max_iter로 설정합니다.
model = HistGradientBoostingClassifier(
    random_state=SEED, 
    max_iter=200,       
    max_depth=8,      
    learning_rate=0.05  
)
model.fit(X_train_scaled, y_train)

val_pred = model.predict(X_val_scaled)
acc = accuracy_score(y_val, val_pred)
print(f"검증 데이터 정확도: {acc:.4f}")

test_pred_proba = model.predict_proba(Xte_scaled)[:, 1]
submission = pd.DataFrame({ID: test[ID], TARGET: test_pred_proba})

submission.to_csv("submission_hgb_proba.csv", index=False)
print("\nsubmission_hgb_proba.csv 생성 완료")

검증 데이터 정확도: 0.8616

submission_hgb_proba.csv 생성 완료
